## Task Overview

This task detects and tracks moving cars in the `Traffic_Laramie_1.mp4` video using motion-based segmentation and a simple tracking method.

The main idea is to focus only on the road area and ignore as much of the surrounding pedestrian and sidewalk region as possible. A polygon-shaped road mask is created for this purpose. Motion is then detected by combining two sources: background subtraction and frame differencing. These two masks are fused so that both persistent motion and short-term motion changes can be captured.

After motion detection, the mask is cleaned using filtering and morphological operations. Contours are extracted from the cleaned foreground regions, and each contour is tested with geometric rules such as area, width, height, aspect ratio, extent, and solidity. These checks help keep vehicle-like objects while removing many non-vehicle blobs.

The remaining detections are merged where necessary and passed to a lightweight tracker. The tracker matches objects by centroid distance from frame to frame, allowing each moving car to remain visible as a continuous tracked object rather than as separate detections in every frame.

Finally, the selected road polygon and the tracked car boxes are drawn on the video output and displayed frame by frame.

In [1]:
import os
import cv2
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple, Dict

VIDEO_PATH = os.path.join('..', 'Exercise1_Files', 'Traffic_Laramie_1.mp4')
if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")



### Importing libraries and locating the input video

This cell imports the libraries needed for video processing, numerical operations, and structured data storage.

The path to the input traffic video is then created. A file existence check is included so that execution stops immediately if the required video is missing. This prevents the later processing steps from failing silently or producing confusing errors.

In [2]:
# Create a binary mask from polygon coordinates so processing stays inside the road area.
def make_polygon_roi_mask(shape, points):
    h, w = shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    pts = np.array(points, dtype=np.int32)
    cv2.fillPoly(mask, [pts], 255)
    return mask

# Clean a foreground mask using blur and morphology to remove noise and fill gaps.
def clean_mask(mask: np.ndarray) -> np.ndarray:
    # Reduce salt-and-pepper style noise.
    mask = cv2.medianBlur(mask, 5)

    # Remove small isolated foreground specks.
    open_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, open_k, iterations=1)

    # Join nearby foreground regions that belong to the same object.
    close_k = cv2.getStructuringElement(cv2.MORPH_RECT, (17, 9))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, close_k, iterations=2)

    # Slightly enlarge the cleaned blobs for more stable contour extraction.
    mask = cv2.dilate(mask, None, iterations=1)
    return mask

# Return the union of two bounding boxes.
def bbox_union(a, b):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    x1 = min(ax, bx)
    y1 = min(ay, by)
    x2 = max(ax + aw, bx + bw)
    y2 = max(ay + ah, by + bh)
    return (x1, y1, x2 - x1, y2 - y1)

# Compute Intersection over Union for two boxes in (x, y, w, h) format.
def iou_xywh(a, b) -> float:
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    ax2, ay2 = ax + aw, ay + ah
    bx2, by2 = bx + bw, by + bh

    ix1, iy1 = max(ax, bx), max(ay, by)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0

# Check whether two boxes are close enough to be considered parts of the same object.
def boxes_are_close(a, b, gap=18):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    ax2, ay2 = ax + aw, ay + ah
    bx2, by2 = bx + bw, by + bh
    return not (ax2 + gap < bx or bx2 + gap < ax or ay2 + gap < by or by2 + gap < ay)

# Merge overlapping or nearby boxes until no more valid merges remain.
def merge_overlapping_boxes(boxes: List[Tuple[int, int, int, int]], iou_merge=0.18):
    boxes = boxes[:]
    changed = True

    while changed:
        changed = False
        out = []
        used = [False] * len(boxes)

        for i in range(len(boxes)):
            if used[i]:
                continue

            cur = boxes[i]
            used[i] = True

            for j in range(i + 1, len(boxes)):
                if used[j]:
                    continue

                # Merge boxes if they overlap enough or lie very close together.
                if iou_xywh(cur, boxes[j]) > iou_merge or boxes_are_close(cur, boxes[j], gap=20):
                    cur = bbox_union(cur, boxes[j])
                    used[j] = True
                    changed = True

            out.append(cur)

        boxes = out

    return boxes

### Defining ROI, mask cleaning, and box-merging helpers

This cell contains the helper functions used to prepare detections before tracking.

`make_polygon_roi_mask()` creates a binary mask from a polygon so that only the road area is analysed. This is important because the scene contains regions that should not contribute to vehicle detection.

`clean_mask()` improves the foreground mask by reducing noise, removing small isolated blobs, joining broken vehicle regions, and slightly expanding the final shapes. This makes the contours more stable.

The remaining functions work with bounding boxes. `bbox_union()` combines two boxes into one larger box, `iou_xywh()` measures overlap strength, and `boxes_are_close()` checks whether two boxes are near enough to be treated as one object. `merge_overlapping_boxes()` uses these rules to merge fragmented detections into more complete vehicle boxes.

In [3]:
# Store the state of one active tracked object.
@dataclass
class Track:
    tid: int
    bbox: Tuple[int, int, int, int]
    cx: float
    cy: float
    missed: int = 0

# Track detections across frames using centroid distance.
class SimpleTracker:
    def __init__(self, dist_max=85, max_missed=12):
        # Maximum allowed centroid distance for a valid match.
        self.dist_max = dist_max

        # Remove tracks that remain unmatched for too many frames.
        self.max_missed = max_missed

        # Assign new IDs incrementally.
        self.next_id = 1

        # Store active tracks by ID.
        self.tracks: Dict[int, Track] = {}

    def update(self, dets: List[Tuple[int, int, int, int]]):
        # Compute detection centroids for matching.
        det_centers = [(x + w / 2.0, y + h / 2.0) for (x, y, w, h) in dets]

        # Build all valid track-detection distance pairings.
        candidates = []
        for tid, tr in self.tracks.items():
            for i, (cx, cy) in enumerate(det_centers):
                d = float(np.hypot(cx - tr.cx, cy - tr.cy))
                if d < self.dist_max:
                    candidates.append((d, tid, i))

        # Match the closest pairs first.
        candidates.sort(key=lambda z: z[0])

        matched_tracks = set()
        matched_dets = set()

        for d, tid, i in candidates:
            if tid in matched_tracks or i in matched_dets:
                continue

            matched_tracks.add(tid)
            matched_dets.add(i)

            x, y, w, h = dets[i]
            tr = self.tracks[tid]
            tr.bbox = (x, y, w, h)
            tr.cx = x + w / 2.0
            tr.cy = y + h / 2.0
            tr.missed = 0

        # Age unmatched tracks and delete stale ones.
        for tid in list(self.tracks.keys()):
            if tid not in matched_tracks:
                self.tracks[tid].missed += 1
                if self.tracks[tid].missed > self.max_missed:
                    del self.tracks[tid]

        # Create new tracks for detections that were not matched.
        for i, (x, y, w, h) in enumerate(dets):
            if i in matched_dets:
                continue
            cx, cy = x + w / 2.0, y + h / 2.0
            self.tracks[self.next_id] = Track(self.next_id, (x, y, w, h), cx, cy)
            self.next_id += 1

### Defining the object tracker

This cell defines the tracking stage.

`Track` stores the information for one active object, including its ID, bounding box, centroid position, and how many frames it has been missed for.

`SimpleTracker` matches new detections to old tracks using centroid distance. First, all possible valid matches are collected. These candidate matches are then sorted by distance so that the closest matches are processed first. When a detection matches an existing track, that track is updated with the new box and new centroid.

Tracks that are not matched are marked as missed, and they are removed if they remain unmatched for too long. Any unmatched detection becomes a new track with a new ID.

This approach is simple but effective when the scene is not too crowded and the object motion is reasonably smooth.

In [4]:
# Decide whether a contour and its bounding box look like a vehicle.
def contour_is_vehicle(c, bbox, roi_mask):
    x, y, w, h = bbox
    area = cv2.contourArea(c)

    # Reject regions that are too small or too large to be plausible cars.
    if area < 1800 or area > 25000:
        return False

    # Reject boxes that are too narrow or too short.
    if w < 42 or h < 24:
        return False

    # Keep mainly horizontal vehicle-like shapes.
    aspect = w / float(h)
    if aspect < 1.0 or aspect > 4.2:
        return False

    # Reject very tall human-like shapes.
    if h > 1.7 * w:
        return False

    # Check how much of the box area is filled by the contour.
    rect_area = w * h
    extent = area / float(rect_area) if rect_area > 0 else 0.0
    if extent < 0.35:
        return False

    # Check contour compactness using solidity.
    hull = cv2.convexHull(c)
    hull_area = cv2.contourArea(hull)
    solidity = area / float(hull_area) if hull_area > 0 else 0.0
    if solidity < 0.65:
        return False

    # Ensure the bottom center of the box lies inside the road ROI.
    fx = int(x + w / 2)
    fy = int(y + h)
    if fy < 0 or fy >= roi_mask.shape[0] or fx < 0 or fx >= roi_mask.shape[1]:
        return False
    if roi_mask[fy, fx] == 0:
        return False

    return True

### Filtering contours to keep vehicle-like objects

This cell defines the contour filtering function.

The aim is to remove many false detections before tracking begins. Each contour is tested using several geometric rules. The contour must have a reasonable area, width, and height. Its aspect ratio should be more consistent with a car than with a pedestrian. Extremely tall shapes are rejected, since they are more likely to belong to people or narrow background fragments.

Two additional measures are used:

- **Extent**, which checks how much of the bounding box is actually filled by the contour.
- **Solidity**, which checks how compact the contour is compared with its convex hull.

Finally, the lower centre of the bounding box must lie inside the road mask. This helps reject objects that overlap the road visually but are not actually positioned on it.

In [5]:
# Open the input video.
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError("Cannot open video")

# Read the first frame so dimensions can be extracted.
ret, first = cap.read()
if not ret:
    raise RuntimeError("Cannot read first frame")

H, W = first.shape[:2]

# Define a road-only polygon to exclude much of the sidewalk and pedestrian region.
roi_poly = [
    (0, int(H * 0.58)),
    (int(W * 0.18), int(H * 0.50)),
    (int(W * 0.88), int(H * 0.50)),
    (W - 1, int(H * 0.62)),
    (W - 1, H - 1),
    (0, H - 1),
]

# Build the binary mask for the selected road polygon.
roi_mask = make_polygon_roi_mask(first.shape, roi_poly)

# Rewind the video so processing starts again from frame zero.
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

# Create the background subtractor for motion segmentation.
bg = cv2.createBackgroundSubtractorMOG2(history=300, varThreshold=22, detectShadows=False)

# Store the previous grayscale frame for frame differencing.
prev_gray = None

# Create the tracker used to follow detected cars across frames.
tracker = SimpleTracker(dist_max=85, max_missed=14)

# Toggle track ID display on or off.
SHOW_IDS = False

# Delay detection slightly so the background model can settle.
WARMUP_FRAMES = 20

# Keep track of the current frame number.
frame_idx = 0

### Preparing the video, ROI, background model, and tracker

This cell sets up the main objects needed before frame-by-frame processing begins.

The video is opened and the first frame is read so that the frame dimensions can be used to define the road polygon. That polygon is then converted into a binary ROI mask, which limits later motion analysis to the road region.

The video position is reset to the start so that no frame is skipped. A MOG2 background subtractor is created for motion segmentation, and a `SimpleTracker` instance is prepared for object tracking.

A short warm-up period is also defined. This allows the background model to stabilise before detections are trusted.

In [ ]:
while True:
    # Read the next frame from the video.
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1

    # Convert the frame to grayscale and smooth it before motion analysis.
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    # Background subtraction highlights persistent moving foreground objects.
    fg = bg.apply(gray)
    _, fg_bin = cv2.threshold(fg, 210, 255, cv2.THRESH_BINARY)

    # Initialize the previous grayscale frame on the first iteration.
    if prev_gray is None:
        prev_gray = gray.copy()

    # Frame differencing highlights short-term motion changes.
    diff = cv2.absdiff(gray, prev_gray)
    prev_gray = gray.copy()
    _, diff_bin = cv2.threshold(diff, 22, 255, cv2.THRESH_BINARY)

    # Fuse both motion cues and restrict them to the road region only.
    fused = cv2.bitwise_or(fg_bin, diff_bin)
    fused = cv2.bitwise_and(fused, roi_mask)
    fused = clean_mask(fused)

    dets = []

    # Start contour-based detection only after the warm-up phase.
    if frame_idx > WARMUP_FRAMES:
        contours, _ = cv2.findContours(fused, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for c in contours:
            bbox = cv2.boundingRect(c)
            if contour_is_vehicle(c, bbox, roi_mask):
                dets.append(bbox)

        # Merge fragmented detections that likely belong to the same car.
        dets = merge_overlapping_boxes(dets, iou_merge=0.18)

    # Update active tracks using the final detections.
    tracker.update(dets)

    # Draw results on a fresh visualization frame.
    vis = frame.copy()

    # Draw the road polygon outline.
    pts = np.array(roi_poly, np.int32)
    cv2.polylines(vis, [pts], True, (0, 255, 255), 2)

    # Draw each tracked car box.
    for tid, tr in tracker.tracks.items():
        x, y, w, h = tr.bbox
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)

        if SHOW_IDS:
            cv2.putText(vis, f"ID {tid}", (x, y - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Show the processed frame in a display window.
    cv2.imshow("Ex1.1 Cars only", vis)

    # Stop early if the q key is pressed.
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

### Main frame-by-frame detection and tracking loop

This cell performs the core video processing.

Each frame is converted to grayscale and smoothed. Motion is then detected in two ways: background subtraction and frame differencing. These two masks are combined so that the detector can respond both to persistent motion and to short-term movement between frames.

The fused mask is restricted to the road region and cleaned with the earlier mask-processing function. After the warm-up phase, contours are extracted and tested with the vehicle filter. Any accepted detections are then merged where needed.

The tracker is updated with the final detection list. A visual output frame is then created by drawing the road polygon and the tracked boxes. Optional ID labels can also be shown if needed.

The result is displayed in a live window, and the loop continues until the video ends or the user presses `q`.

In [ ]:
# Release the video handle and close all OpenCV windows.
cap.release()
cv2.destroyAllWindows()

### Releasing resources at the end

This final cell closes the video stream and destroys any display windows created by OpenCV.

This cleanup step ensures that file handles and GUI resources are released properly after processing finishes.